Introduction
============

This worksheet **explains the basic concepts of path planning, work and configuration space** needed for the course  "Introduction to robot path planning"

License is based on Creative Commons: Attribution-NonCommercial 4.0 International (CC BY-NC 4.0) (pls. check: http://creativecommons.org/licenses/by-nc/4.0/)


Version | Author
------------ | -------------
0.2. | Björn Hein



In [ ]:
# needed for the figures in this notebook (drawing functions: see IPVISIntroduction.py)
import itertools
import numpy as np
import matplotlib.pyplot as plt
from shapely.geometry import Point, Polygon, LineString, box

import IPVISIntroduction as vis
%matplotlib inline

# Path / Motion Planning

* What is path / motion planning?
* Work space $\leftrightarrow$ configuration space

### The Piano Mover's Problem

* Piano is a "robot" with 3 **DoF (Degrees of Freedom)**. Moving in three axis: translation in X-axis, in Y-axis, and rotation around Z-axis. 
* The kinematics is holonomic. 
  
  **Remark - holonomic**: Every DoF can be changed separately and independently from the others. E.g. Movement along X-axis can be done independently from the movement along the Y-axis or the rotation around the Z-axis
  
  **Remark - non-holonomic**: a typical example is a car. It is not possible to turn without moving forward.
.
* Wall, chairs, tables, etc…. are fixed. We only use static obstacles.

Especially two interesting problems to solve regarding path planning:
* Getting a path, consisting of collision free movements between start and goal position.
* Information, whether such a path doesn‘t exist.

In [ ]:
# top view of a room: walls with a door and some furniture (units: m)
room = dict()
room["walls"]     = box(0, 0, 10, 0.15).union(box(0, 6.85, 10, 7)).union(box(0, 0, 0.15, 7)).union(box(9.85, 0, 10, 7))
room["innerWall"] = box(5.9, 0, 6.1, 2.6).union(box(5.9, 4.4, 6.1, 7))   # with a door, 1.8 m wide
room["table"]     = box(1.6, 4.6, 3.4, 5.6)
room["sofa"]      = box(7.4, 0.15, 9.85, 1.1)
room["shelf"]     = box(7.0, 6.4, 9.6, 6.85)

# configurations q = (x, y, theta) the piano passes on its way from q_s to q_g
waypoints = [(2.3, 2.0, 0.0), (4.2, 3.5, -np.pi / 2), (7.8, 3.5, -np.pi / 2), (8.2, 4.4, -np.pi / 3)]
vis.plotPianoMovers(room, waypoints, roomLimits=((0, 10), (0, 7)))

The piano has to be rotated to get through the door: a path for this "robot" needs all three DoF. Poses that would collide with walls or furniture are drawn in orange (change the `waypoints` and try it).

### Configuration Space (C-Space)

Before starting with path planning algorithms, it would be good to have a common representation of the  problem description for different planning tasks. In this way we will be able to compare the difficulty of path planning task and will be able to evaluate the performance of path planning algorithms $\rightarrow$ **C-Space**

* C-Space is a Tool to describe planning tasks for different types of problems in a common way [Udupa 77, Lozano-Perez 83]

* A **configuration** is a set of independent parameters uniquely specifying the actual state of a robotic system e.g.:
  * the values of the 6 axes of an industrial robot. E.g. represented by the 6 dimensional tuple
  
    **[joint1, joint2, joint3, joint4, joint 5, joint 6]**
    
  * if the robot is a “disc“ and its configuration is specified by the 2 dimensional tuple
  
    **[x,y]**
  
  * if the robot is a piano the coords x,y **and** the orientation around z-axis 
  
    **[$t_x$, $t_y$, $r_z$]**

* **Configuration Space** = space of all configurations



### Example: "point" robot

* Goal: transfer „point“ robot from start $q_s$ to goal $q_g$
* as the robot has no dilatation in W-space --> C-space ≡ W-space
* DoF = 2:  C = R²

In [ ]:
# W-space with two obstacles and a narrow passage
scene = dict()
scene["left"]  = box(0, 8, 9, 14).union(box(9, 9, 10, 13)).union(box(10, 10, 10.6, 12))
scene["right"] = box(13, 8, 22, 14).union(box(12, 9, 13, 13)).union(box(11.4, 10, 12, 12))

path = [(3, 19), (11, 15), (11, 7), (3, 3)]   # from q_s to q_g through the passage
vis.plotPointRobot(scene, path)

As it is C-space ≡ W-space in this example. The idea now is to transform **all** planning problems into a problem that looks like planning a point robot.

### A "Disc-Robot" = point robot with dilatation

A Disc-Robot is basically a point robot with an additional radius. Therefore following applies

* disc robot = point robot + radius
* DoF = 2:  C = R²
* **BUT**: C-space ≠ W-space

$A(q)$: subset of the W-space occupied by the robot $A$ at configuration $q$

In [ ]:
vis.plotDiscRobot(q=(6, 5), radius=2)

Main question is then, how to transform the W-Space into the C-space. Because then we can again plan a point robot in C-Space

In the case of the disc robot the transformation is pretty straight forward. Add the radius to every obstacle. See the following:

In [ ]:
obstacle = Polygon([(2, 6), (6, 6), (4, 2)])   # obstacle B_i in W-space
vis.plotMinkowskiSum(obstacle, radius=1.0)

W-Space obstacle $B_i$ + the radius of the Robot results in the obstacle representation $CB_i$ in C-Space. On the right the disc is placed at several positions where it just touches the obstacle: its center (= its configuration) lies on the border of $CB_i$. Therefore the corners of $CB_i$ are rounded. (Mathematically this is the Minkowski sum of obstacle and robot; with shapely it is simply `obstacle.buffer(radius)`.)

Instead of planning the disc-robot in W-space it is now possible to plan a point robot in the corresponding C-space. Following another example.

In [ ]:
maze = dict()
maze["left"]   = box(3, 2, 5, 17)
maze["bottom"] = box(3, 2, 16, 4)
maze["right"]  = box(14, 2, 16, 20)
maze["upper"]  = box(5, 13, 10, 15)
maze["lower"]  = box(8, 8, 14, 10)

robotRadius = 0.9
vis.plotDiscRobotCSpace(maze, robotRadius, start=(9, 20), goal=(10, 6))

Left: Disc-robot with obstacles, right: obstacles transformed into C-Space by extending them with the radius of the disc robot. **Now it is possible to plan a point robot solving the disc robot problem on the left**.

**Try it:** set `robotRadius = 1.6` and run the cell again. The C-obstacles grow, the passages close and there is no path any more.

### Example: planar robot (2Dof)

Following example shows a robot with two rotational joints. To describe the **configuration** of this robot a two dimensional tupel is used: [$j_1$, $j_2$]. Again, the idea is to use a point robot to solve the planning problem. Therefore the obstacles in W-Space have to be transformed into the C-Space. This is done by sampling and discretizing of the C-Space and checking for every **configuration**, whether it collides or not.

First the robot and the obstacles in W-space:

In [ ]:
from IPPlanarManipulator import PlanarRobot
from IPEnvironmentKin import KinChainCollisionChecker

# W-space: three obstacles (same scene as in IP-10-1)
obstacles = dict()
obstacles["obstacle 1"] = LineString([(-2, 0), (-0.8, 0)]).buffer(0.5)
obstacles["obstacle 2"] = LineString([(2, 0), (2, 1)]).buffer(0.2)
obstacles["obstacle 3"] = LineString([(-1, 2), (1, 2)]).buffer(0.1)

robot2 = PlanarRobot(n_joints=2)   # two links, each 1.5 long
jointLimits2 = [[-np.pi, np.pi], [-np.pi, np.pi]]

Now the C-space is sampled: every joint is discretized, and for every resulting configuration the robot is moved there and checked for collision in W-space. To see later, which obstacle caused the collision, there is one collision checker per obstacle.

In [ ]:
def sampleCSpace(robot, obstacles, limits, resolution):
    """Discretize every joint with `resolution` values and check every configuration for collision.
    Result: cspace[i, j, ...] = 0 if the configuration is free, k if it collides with the k-th obstacle."""
    checkers = [KinChainCollisionChecker(robot, {name: obstacle}, limits=limits)
                for name, obstacle in obstacles.items()]
    jointValues = [np.linspace(low, high, resolution) for low, high in limits]
    cspace = np.zeros([resolution] * len(limits), dtype=int)
    for index in itertools.product(range(resolution), repeat=len(limits)):
        q = [jointValues[d][i] for d, i in enumerate(index)]
        for k, checker in enumerate(checkers, start=1):
            if checker.pointInCollision(q):   # collision check in W-space
                cspace[index] = k
                break
    return jointValues, cspace

jointValues2, cspace2 = sampleCSpace(robot2, obstacles, jointLimits2, resolution=100)
print(f"{cspace2.size} configurations checked, {np.count_nonzero(cspace2)} of them in collision")

If all obstacles are transformed, the path planning can take place in the C-Space using a point robot (here: shortest path on the free grid cells). The result solves the path planning problem of the robot in W-space:

In [ ]:
start2, goal2 = [2.0, 0.5], [-2.0, -0.5]
path2 = vis.gridPath(jointValues2, cspace2, start2, goal2)
vis.plotPlanarRobotCSpace(robot2, obstacles, jointValues2, cspace2, path2)

Left: the robot at start $q_s$ (green), at goal $q_g$ (red) and at some configurations in between (gray). Right: the C-space. Every colored cell is a configuration, in which the robot collides with the obstacle of the same color. The black line is the path of the point robot in C-space, the gray dots belong to the gray robot poses on the left.

**Remark:** Joint angles are periodic, $-\pi$ and $\pi$ are the same angle. So the left and right border (as well as the lower and upper border) of the C-space belong together, strictly speaking this C-space is a torus.

**Try it** (optional, needs `ipywidgets`): move the robot with the sliders and watch where its configuration lies in the C-space.

In [ ]:
from ipywidgets import interact, FloatSlider

def showConfiguration(j1, j2):
    vis.plotConfiguration(robot2, obstacles, jointValues2, cspace2, [j1, j2])

interact(showConfiguration,
         j1=FloatSlider(min=-np.pi, max=np.pi, step=0.05, value=2.0),
         j2=FloatSlider(min=-np.pi, max=np.pi, step=0.05, value=0.5));

**Remark:** As you can see, it is possible to transform planning problems of more complex robots into planning problems using a simple point robot. If this is possible for all robots, we can just develop algorithms for point robots and then use them to solve problems for more complex robotic systems.

**Summary**: We will focus on developping algorithm for point robots as they can be applied for arbitrary robotic systems.

### Example: planar robot (3 Dof)

Following an example of a robot with three rotational joints. The **configuration** of this robot is described by the tuple [$j_1$, $j_2$, $j_3$]. The C-space is sampled with the same function as above, it is now 3 dimensional (this takes a few seconds). It is obvious that even simple looking problems in W-Space can be difficult problems to solve.

In [ ]:
obstacles3 = dict()
obstacles3["obstacle 1"] = Point(-0.8, 2.2).buffer(0.6)
obstacles3["obstacle 2"] = Point(2.2, 1.2).buffer(0.6)
obstacles3["obstacle 3"] = Point(-1.8, -2.0).buffer(0.6)

robot3 = PlanarRobot(n_joints=3)
jointLimits3 = [[-np.pi, np.pi]] * 3
jointValues3, cspace3 = sampleCSpace(robot3, obstacles3, jointLimits3, resolution=32)
print(f"{cspace3.size} configurations checked, {np.count_nonzero(cspace3)} of them in collision")

vis.plotPlanarRobotCSpace3D(robot3, obstacles3, jointValues3, cspace3, start=[-0.6, 0.8, 0.8], goal=[2.4, 0.6, 0.6])

Again, it is possible to transform the path planning problem to solve it using a point robot in C-space.

### Challenges/Problems planning in C-Space

In general, as we can see, doing path planning in C-space is a very nice concept as it allows to focus on developing algorithms for point robots. **However it has one major downside**: In our concept for now we have to transform the obstacles from W-Space to C-Space before we can do the planning. Following table lists the needed time and memory to transform the W-Space for a 6-Dof-Robot.

| DoF | 6 | 6 | 6 |
| :--- | ---: | ---: | ---: |
| Discretization | [1, 1, 1, 1, 1, 1]° | [2, 2, 3, 6, 6, 20]° | [10, 10, 10, 10, 10, 10]° |
| # of configurations | $1.98 \cdot 10^{15}$ | $2.29 \cdot 10^{11}$ | $1.98 \cdot 10^{9}$ |
| time for calculation (per configuration) | 1 ms | 1 ms | 1 ms |
| overall time | 62654 years | 7 years | 23 days |
| memory per configuration | 1 bit | 1 bit | 1 bit |
| memory usage | 225 TByte | 27 GByte | 236 MByte |

**It is obvious, that it is not possible to use this for real world problems!** It makes no sense to wait in the fastest case 23 days for transforming the W-Space to C-Space before being able to plan the path.

As we still like to plan in the C-Space we have to do it in another way. We use **Planning in the implicit C-Space**

### Implicit C-Space

On the one hand, as shown in the previous paragraph, it is not reasonably possible to transform the W-Space into C-Space for real world problems. On the other hand, we want to use this flexible idea of doing the planning in the C-Space.

The solution is to use the **implicit C-Space** planning. The idea is pretty simple:

1. Planning is done in the C-Space
2. Obstacles are detected **during** the search using collision detection in W-Space

The following example uses the 2-DoF robot from above and a simple search, that always continues at the configuration closest to the goal. Every configuration it reaches is checked for collision in W-space; nothing else is known about the obstacles.

In [ ]:
environment2 = KinChainCollisionChecker(robot2, obstacles, limits=jointLimits2)
result = vis.implicitSearch(environment2, start2, goal2, stepSize=0.2, maxChecks=250)
vis.plotImplicitCSpace(robot2, obstacles, jointValues2, cspace2, result)

Left: the planner only knows the configurations it has checked so far (blue: free, orange: collision). The dashed lines show the C-obstacles computed above; they are only drawn for comparison, the planner does not know them. Right: for every configuration the robot is placed in W-space and checked for collision (orange: last configuration in collision, blue: current end of the path).

**Try it:** reduce `maxChecks` (e.g. to 60) to see an earlier state of the search, or change `goal2`.

**Summary**: The solution is pretty simple. We keep our concept and do the planning in the C-Space. Every **configuration** we reach in C-Space **during our search** we check for collision in W-Space.

**Remark**: The approach **exploring during search** has one important implication. As we are exploring the C-Space during the search the path planning algorithms actually don't know where the obstacles are. So they cannot use this information to optimize the planning. Strategies more or less have to consider concepts, how to search in an efficient, when you are basically **blind**.

# Testing environments



Shown in the previous section it is possible to stick to the concept of planning a C-Space (= planning a point robot). Therefore to dive into the world of path planning we first define test environments, in which we will test our planning algorithms. 

